# Finetune Expert 2 MGA-CLAP trên Google Colab

Notebook này chạy `Train_Expert2.py` hiện có, dùng cosine trực tiếp. Chọn **Runtime → Change runtime type → GPU** trước khi chạy.

Chuẩn bị trên Google Drive:

- `MyDrive/MOESCORE/`: bản sao thư mục code hiện tại, gồm `Train_Expert2.py`, các file `expert2_*.py`, `configs/`, **toàn bộ `MGA-CLAP/` đã có các patch trong repo**, `requirements-expert2.txt` và `weights/mga-clap.pt`. Không dùng lại bản MGA-CLAP gốc chưa được patch.
- `MyDrive/XACLE_dataset.zip`: một file ZIP chứa nguyên cấu trúc dataset, trong đó có `meta_data/train_average.csv`, `meta_data/validation_average.csv`, `wav/train/` và `wav/validation/`. ZIP có thể chứa một thư mục bao ngoài; notebook tự tìm data root. Nếu đã tải dataset thành thư mục trên Drive, đặt ở `MyDrive/XACLE_dataset/` và bỏ qua ZIP.

ZIP được giải nén vào `/content`, là ổ tạm của Colab. Best checkpoint được cập nhật khi validation SRCC cải thiện; `last.pt` được ghi vào Drive sau mỗi epoch. Drive cần đủ dung lượng cho các checkpoint lớn; Colab có thể ngắt phiên giữa epoch, khi đó chạy lại notebook và đặt `RESUME = True` ở cell train. Đọc [Colab FAQ](https://research.google.com/colaboratory/faq.html) về giới hạn GPU, thời gian phiên và I/O của Drive.


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

PROJECT_DIR = Path('/content/drive/MyDrive/MOESCORE')
DATA_ARCHIVE = Path('/content/drive/MyDrive/XACLE_dataset.zip')
DATA_FOLDER = Path('/content/drive/MyDrive/XACLE_dataset')
OUTPUT = PROJECT_DIR / 'weights/expert2_best.pt'

required = [
    'Train_Expert2.py', 'expert2_data.py', 'expert2_loss.py', 'expert2_model.py',
    'configs/expert2_finetune.yaml', 'requirements-expert2.txt',
    'MGA-CLAP/settings/inference_example.yaml', 'MGA-CLAP/models/ase_model.py',
    'MGA-CLAP/models/htsat.py', 'MGA-CLAP/models/text_encoder.py',
    'weights/mga-clap.pt',
]
missing = [name for name in required if not (PROJECT_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Thiếu file trong {PROJECT_DIR}: {missing}')
print('Project:', PROJECT_DIR)
print('MGA checkpoint:', PROJECT_DIR / 'weights/mga-clap.pt')


In [ ]:
import zipfile

if DATA_ARCHIVE.is_file():
    staging = Path('/content/xacle_data')
    marker = staging / '.extract_complete'
    if not marker.is_file():
        staging.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(DATA_ARCHIVE) as archive:
            archive.extractall(staging)
        marker.touch()
    search_root = staging
elif DATA_FOLDER.is_dir():
    search_root = DATA_FOLDER
else:
    raise FileNotFoundError(f'Cần {DATA_ARCHIVE} hoặc {DATA_FOLDER}')

matches = list(search_root.rglob('meta_data/train_average.csv'))
if len(matches) != 1:
    raise RuntimeError(f'Cần đúng một train_average.csv, tìm thấy {len(matches)} trong {search_root}')
DATA_ROOT = matches[0].parent.parent
for name in ['meta_data/validation_average.csv', 'wav/train', 'wav/validation']:
    if not (DATA_ROOT / name).exists():
        raise FileNotFoundError(DATA_ROOT / name)
print('Data root:', DATA_ROOT)


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                str(PROJECT_DIR / 'requirements-expert2.txt')], check=True)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Không thấy GPU. Chọn Runtime → Change runtime type → GPU, rồi chạy lại.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))


In [ ]:
import yaml

with (PROJECT_DIR / 'configs/expert2_finetune.yaml').open(encoding='utf-8') as stream:
    config = yaml.safe_load(stream)
config.update({
    'data_root': str(DATA_ROOT),
    'mga_source': str(PROJECT_DIR / 'MGA-CLAP'),
    'mga_config': str(PROJECT_DIR / 'MGA-CLAP/settings/inference_example.yaml'),
    'mga_checkpoint': str(PROJECT_DIR / 'weights/mga-clap.pt'),
    'output': str(OUTPUT),
    'device': 'cuda',
})
COLAB_CONFIG = Path('/content/expert2_finetune_colab.yaml')
COLAB_CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print('Config:', COLAB_CONFIG)
print('Batch size:', config['batch_size'], '| epochs:', config['epochs'])

subprocess.run([sys.executable, '-u', str(PROJECT_DIR / 'Train_Expert2.py'),
                '--config', str(COLAB_CONFIG), '--check-data'],
               cwd=PROJECT_DIR, check=True)


## Huấn luyện hoặc tiếp tục

Chạy lần đầu với `RESUME = False`. Nếu Colab ngắt phiên, chạy lại các cell từ đầu, đổi thành `RESUME = True` và chạy cell dưới. `last.pt` chỉ được ghi sau **mỗi epoch**, nên phần epoch bị ngắt sẽ chạy lại. Giữ cùng `epochs`, batch size và cấu hình khi resume để lịch learning rate nhất quán. Batch mặc định là 8 theo cấu hình thí nghiệm; nếu GPU hết VRAM, giảm batch trong YAML trước khi tạo config sẽ làm thay đổi số cặp của pairwise loss, cần ghi nhận là thí nghiệm khác.


In [ ]:
RESUME = False  # Đổi thành True sau khi đã có last.pt từ lần chạy trước.
LAST = OUTPUT.parent / 'expert2_run/last.pt'
if LAST.exists() and not RESUME:
    raise RuntimeError(f'Đã có {LAST}; đặt RESUME = True để tiếp tục.')
if RESUME and not LAST.is_file():
    raise FileNotFoundError(LAST)

cmd = [sys.executable, '-u', str(PROJECT_DIR / 'Train_Expert2.py'),
       '--config', str(COLAB_CONFIG)]
if RESUME:
    cmd += ['--resume', str(LAST)]
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


In [ ]:
import json

RUN_DIR = OUTPUT.parent / 'expert2_run'
METRICS = RUN_DIR / 'best_metrics.json'
if METRICS.is_file():
    print(json.dumps(json.loads(METRICS.read_text(encoding='utf-8')), indent=2))
    print('Best checkpoint:', OUTPUT)
    print('Validation predictions:', RUN_DIR / 'validation_predictions.csv')
else:
    print('Chưa có kết quả validation; xem log của cell train.')


## Đánh giá checkpoint tốt nhất trên test

Chạy sau khi đã chọn `expert2_best.pt` bằng validation SRCC. Cell này đọc `test_average.csv` và `wav/test/`, tính SRCC/LCC/KTAU/MSE rồi lưu metric và prediction trong `weights/expert2_run/`. Không dùng kết quả test để chọn lại checkpoint.


In [ ]:
if not OUTPUT.is_file():
    raise FileNotFoundError(f'Chưa có best checkpoint: {OUTPUT}')
for name in ['meta_data/test_average.csv', 'wav/test']:
    if not (DATA_ROOT / name).exists():
        raise FileNotFoundError(DATA_ROOT / name)
subprocess.run([sys.executable, '-u', str(PROJECT_DIR / 'Train_Expert2.py'),
                '--config', str(COLAB_CONFIG), '--eval-only', '--split', 'test',
                '--checkpoint', str(OUTPUT)], cwd=PROJECT_DIR, check=True)
TEST_METRICS = OUTPUT.parent / 'expert2_run/test_evaluation_metrics.json'
print('Test metrics:', TEST_METRICS)
print('Test predictions:', OUTPUT.parent / 'expert2_run/test_evaluation_predictions.csv')
